# Attitude agility — how it works

This is the explanation and reference for the `quicksat` agility budget: what the config file holds, how a slew time falls out of two numbers, why the spacecraft's mass properties arrive as named cases, and where the tool stops.

It is deliberately not a tutorial. In [Diátaxis](https://diataxis.fr/) terms, `sample/` holds the tutorials and how-to guides; `docs/` holds the explanation and the reference. It still runs, because an explanation that cannot be executed drifts from the code it describes. It runs against `docs/data/`, its own copy of the sample satellite's input files. 

The premise throughout: **a sizing model wants to know whether 40 degrees fits, not how the command is shared between wheels.** The whole wheel geometry collapses into two numbers per axis — how much momentum and how much torque the assembly can put about it — and the slew arithmetic takes it from there. There is no distribution matrix and no per-wheel loading.

In [1]:
import os
from pathlib import Path

import pandas as pd

# Make the in-development quicksat package importable without installing it: walk up
# from the current directory to the repo root (the folder that holds the quicksat
# package) and switch to it. Works whether the notebook runs from docs/, the repo
# root, or the docs build.
here = Path.cwd()
repo_root = next(
    (p for p in (here, *here.parents) if (p / "quicksat" / "__init__.py").exists()),
    None,
)
if repo_root is None:
    raise RuntimeError("could not locate the quicksat repo root")
os.chdir(repo_root)

from quicksat import u
from quicksat.agility.budget import AgilityBudget, AgilityConfig, Axis
from quicksat.mass.budget import MassBudget
from quicksat.utils.mission import Mission

pd.set_option("display.float_format", lambda value: f"{value:,.3f}")



In [2]:
DATA = Path("docs") / "data"
config = AgilityConfig.from_yaml_file(DATA / "agility_config.yaml")
mission = Mission.from_yaml_file(DATA / "mission.yaml")
mass_data = MassBudget.from_csv(DATA / "equipment.csv", DATA / "mass_budget_config.yaml")

roll = AgilityBudget(config, mission, Axis.ROLL, "first_guess", mass_budget=mass_data)

## The config file

Three sections, and two deliberate absences.

| section | holds |
|---|---|
| `inertia_cases` | named sets of mass properties; pick one at the call |
| `wheels` | the pyramid, its nameplate figures, and how much of them policy allows |
| `settling_time` | what the platform needs after a slew before the payload can work |

**No mass.** The mass budget computes it, and a second copy here would drift — the same argument that keeps altitude out of every config but `mission.yaml`. It reaches the budget through an attached `MassBudget`, read at the case's own propellant fraction.

**No target duration.** How long a manoeuvre may take is a question asked *of* a spacecraft, not a property *of* one: the same platform is asked it differently by a routine acquisition and by an emergency reslew. It arrives as an argument.

In [3]:
print(Path("docs/data/agility_config.yaml").read_text())

# Attitude agility settings: the spacecraft cases that can be slewed, the wheels
# that slew them, and how long the platform takes to settle afterwards.
#
# No mass anywhere. The mass budget owns it, and a second copy here would drift --
# the same reason altitude lives only in mission.yaml. Orbit comes from mission.yaml,
# for ground track speed. The target duration for a manoeuvre is not here
# either: that is asked of a spacecraft, not a property of one.

inertia_cases:
  # One case is one spacecraft, one set of mass properties. The names are yours
  # -- nothing in the code matches on them. Each case is EITHER an envelope
  # estimate OR a stated inertia, never both, so there is never a question which
  # of the two produced a number.

  first_guess:               # inertia estimated from a uniform box
    body:
      x: 1.5 m               # along track
      y: 1.5 m               # cross track
      z: 2.0 m               # nadir
    appendage_factor:        # uplift for arrays,

## Inertia cases, and why the two shapes are kept apart

A case is one spacecraft, one set of mass properties. It is **either** an envelope estimate — a box and a per-axis appendage uplift — **or** a stated inertia, as a mass properties report or a CAD model gives it. Never both, and never neither; a model validator says so on load and names the keys it found, so a typo in `appendage_factor` reads as a missing envelope rather than as a case matching no shape at all.

The reason to keep them apart is what happens when the spacecraft changes. An envelope inertia is *derived* from the mass, so it follows the mass budget automatically. A stated inertia is a fixed number that does not, and nothing can tell you when the two have parted company. Mixing them inside one case would make it impossible to say which behaviour you had.

The names are the user's. Nothing in the code matches on them — `for_case` is a dictionary lookup, and an unlisted name is an error rather than a silent fallback, which would quietly answer for a spacecraft nobody asked about.

In [4]:
for name, case in config.inertia_cases.items():
    budget = AgilityBudget(config, mission, Axis.ROLL, name, mass_budget=mass_data)
    body, uplift = case.body, case.appendage_factor
    if body is not None and uplift is not None:
        shape = "envelope"
        detail = (
            f"box {body.x:~.1f} x {body.y:~.1f} x {body.z:~.1f}, "
            f"factor {uplift.roll}, at {case.propellant:~.0f} propellant"
        )
    else:
        shape, detail = "stated", "stated outright; no mass involved"
    print(f"{name:14s} {shape:9s} {budget.inertia:~8.1f}   {detail}")

try:
    config.for_case("no_such_case")
except KeyError as error:
    print(f"\nunknown name -> {error}")

first_guess    envelope     262.3 kg * m ** 2   box 1.5 m x 1.5 m x 2.0 m, factor 1.07, at 100 % propellant
measured_bol   stated       264.7 kg * m ** 2   stated outright; no mass involved
measured_eol   stated       250.0 kg * m ** 2   stated outright; no mass involved

unknown name -> "No inertia case named 'no_such_case'. Configured: first_guess, measured_bol, measured_eol"


### Only an envelope case needs a mass

Inertia is the single route the mass takes into the slew arithmetic — rate is momentum over inertia, acceleration is torque over inertia, and nothing else in the model looks at the spacecraft at all. So a case that states its inertia needs no mass, no mass budget, and no propellant fraction. Asking one for its mass is a question with no answer, and an envelope case with nothing to weigh says so rather than guessing.

In [5]:
stated = AgilityBudget(config, mission, Axis.ROLL, "measured_bol")
print(f"a stated case, with no mass budget at all:  {stated.inertia:~.1f}")
print(f"  and it still slews:  40 deg in {stated.slew_time(40 * u.deg):~.1f}")

orphan = AgilityBudget(config, mission, Axis.ROLL, "first_guess")
try:
    _ = orphan.inertia
except ValueError as error:
    print(f"\nan envelope case with nothing to weigh:\n  {error}")

a stated case, with no mass budget at all:  264.7 kg * m ** 2
  and it still slews:  40 deg in 63.7 s

an envelope case with nothing to weigh:
  Case 'first_guess' estimates its inertia from an envelope, so it needs a mass. Attach a mass_budget, pass mass=, or use a case that states its inertia outright.


### The propellant fraction lives in the case

It only means something while a mass is being *scaled*, so it belongs to envelope cases and is simply absent from stated ones. That also keeps it off the call signature: choosing a mission point is choosing a case.

The effect is real and it runs the friendly way. Tanks empty over a mission, so the spacecraft gets lighter, turns faster and reaches further inside the same time. Agility is one of the few budgets that improves with age.

In [6]:
rows = []
for name in ("first_guess", "measured_bol", "measured_eol"):
    budget = AgilityBudget(config, mission, Axis.ROLL, name, mass_budget=mass_data)
    rows.append({
        "case": name,
        "inertia": budget.inertia.magnitude,
        "max_rate": budget.max_rate().magnitude,
        "slew_40_deg": budget.slew_time(40 * u.deg).magnitude,
        "reach_in_113_s": budget.achievable_angle(113 * u.s).magnitude,
    })
pd.DataFrame(rows)

,case,inertia,max_rate,slew_40_deg,reach_in_113_s
0,first_guess,262.273,0.737,63.186,61.960
1,measured_bol,264.700,0.730,63.689,61.392
2,measured_eol,250.000,0.773,60.645,65.001


## The wheel geometry collapses to a projection

Four wheels on a pyramid whose symmetry axis is along yaw. That mounting puts **roll and pitch both in the base plane**, which is why one implementation serves both axes and takes the axis as an argument: the geometry is identical and only the inertia differs.

Each wheel contributes `cos(elevation)` of itself to the base plane, and `cos(45°)` of that to any one in-plane axis. Multiply by the number of wheels and you have the axis capability:

$$H_{axis} = n \cdot H_{wheel} \cdot \cos(\epsilon)\cos(45°) \qquad T_{axis} = n \cdot T_{wheel} \cdot \cos(\epsilon)\cos(45°)$$

Yaw sees `sin(elevation)` instead, and comes out the weak axis under this mounting. It drives no slew case here, but it is reported rather than dropped, because it is what a yaw manoeuvre would have to live within.

The **momentum use factor** and the **torque derating** are policy, not hardware. What is held back is disturbance storage and control authority during the slew — which is why they sit in config beside the wheels rather than being folded into the nameplate figures the vendor quotes.

In [7]:
wheels = config.wheels
print(f"nameplate, per wheel   {wheels.momentum:~.1f}   {wheels.torque:~.2f}")
print(f"policy allows          {wheels.momentum_use_factor:~.1f} of momentum, "
      f"{wheels.torque_derating:~.0f} of torque")
print(f"so usable per wheel    {roll.usable_momentum_per_wheel:~.3f}   "
      f"{roll.usable_torque_per_wheel:~.3f}")
print()
print(f"projection             cos({wheels.elevation:~.1f}) x cos(45 deg) = "
      f"{roll.projection.magnitude:.4f}")
print(f"about an in-plane axis {roll.axis_momentum():~.3f}   {roll.axis_torque():~.4f}")
print(f"about yaw              {roll.yaw_momentum:~.3f}   -- the weak axis")

nameplate, per wheel   4.0 m * N * s   0.20 m * N
policy allows          33.3 % of momentum, 75 % of torque
so usable per wheel    1.332 m * N * s   0.150 m * N

projection             cos(26.5 deg) x cos(45 deg) = 0.6328
about an in-plane axis 3.372 m * N * s   0.3797 m * N
about yaw              2.377 m * N * s   -- the weak axis


### One failed wheel halves the axis

With one of four gone, only **two of the three survivors** can be driven at full torque if the net in-plane momentum is to stay zero. Both the momentum and the torque about the axis halve.

Note what does *not* change: the geometry and the projection. This is the same pyramid flying degraded, not an orthogonal three-wheel mounting. Rate and acceleration both halve, and since the crossover angle is `ω²/α`, it halves too — so a degraded spacecraft becomes momentum limited at smaller angles as well as slower at all of them.

In [8]:
print(f"{'':22s}{'nominal':>12s}{'one failed':>12s}")
for label, nominal, degraded in (
    ("axis momentum", roll.axis_momentum(), roll.axis_momentum(degraded=True)),
    ("axis torque", roll.axis_torque(), roll.axis_torque(degraded=True)),
    ("max rate", roll.max_rate(), roll.max_rate(degraded=True)),
    ("max acceleration", roll.max_acceleration(), roll.max_acceleration(degraded=True)),
    ("crossover angle", roll.crossover_angle(), roll.crossover_angle(degraded=True)),
):
    print(f"  {label:20s}{nominal.magnitude:12.4f}{degraded.magnitude:12.4f}")

                           nominal  one failed
  axis momentum             3.3716      1.6858
  axis torque               0.3797      0.1898
  max rate                  0.7366      0.3683
  max acceleration          0.0829      0.0415
  crossover angle           6.5407      3.2703


## Two regimes, and the angle where they swap

$$\omega_{max} = \frac{H_{axis}}{I} \qquad \alpha_{max} = \frac{T_{axis}}{I} \qquad \theta_{cross} = \frac{\omega_{max}^2}{\alpha_{max}}$$

Below the crossover the wheels never saturate. The slew accelerates to the halfway point and decelerates to the end — a **triangular** rate profile, torque limited:

$$t = 2\sqrt{\theta / \alpha} \qquad \omega_{peak} = \sqrt{\theta\alpha}$$

Above it the wheels hit their momentum limit before the midpoint, so the slew coasts — a **trapezoid**, momentum limited:

$$t = \frac{\theta}{\omega_{max}} + \frac{\omega_{max}}{\alpha} \qquad \omega_{peak} = \omega_{max}$$

Reporting which of the two binds is the point rather than a detail: it tells you whether more torque or more momentum would buy anything. Below the crossover, faster wheels help. Above it, they do not — only bigger ones.

Momentum used makes the same statement as a number. It reaches 1 for every slew past the crossover, which is what momentum limited *means*; below it, the shortfall is headroom a larger slew would spend.

In [9]:
crossover = roll.crossover_angle()
angles = [crossover * 0.5, crossover * 0.99, crossover * 1.01, crossover * 5]
print(f"crossover at {crossover:~.2f}\n")
print(f"{'angle':>10}{'profile':>14}{'time':>10}{'peak rate':>12}{'momentum':>11}")
for angle in angles:
    print(f"{angle.magnitude:9.2f}{roll.profile(angle).value:>14}"
          f"{roll.slew_time(angle).magnitude:9.1f}s"
          f"{roll.peak_rate(angle).magnitude:11.4f}"
          f"{roll.momentum_used(angle).magnitude:10.1%}")

crossover at 6.54 deg

     angle       profile      time   peak rate   momentum
     3.27    triangular     12.6s     0.5208     70.7%
     6.48    triangular     17.7s     0.7329     99.5%
     6.61   trapezoidal     17.8s     0.7366    100.0%
    32.70   trapezoidal     53.3s     0.7366    100.0%


## The target duration is an argument, not config

`settling_time` is config, because it is a property of the platform: the ADCS needs that long to damp out before the payload can work, whatever the manoeuvre. It is applied once at the end, not between steps.

The time *allowed* is different in kind. The same spacecraft is asked for 40° in 113 seconds by one operation and in 90 by another, and neither is more true than the other. So `time_margin`, `achievable_angle` and `tabulated_agility` all take a `target_duration`, and the budget answers whichever question you ask.

`achievable_angle` is the inverse solve — the same arithmetic run backwards — so it has to agree with `slew_time` exactly, and there is a test that holds it to that.

In [10]:
angle = 40 * u.deg
for target in (113 * u.s, 90 * u.s, 70 * u.s):
    margin = roll.time_margin(angle, target)
    reach = roll.achievable_angle(target)
    verdict = "fits" if margin.magnitude >= 0 else "does not fit"
    print(f"in {target:~5.0f}:  {angle:~.0f} needs {roll.total_time(angle):~.1f}, "
          f"{margin.to('percent'):~+6.1f} -- {verdict:12s} (can reach {reach:~.1f} within the allocated duration)")

print("\nthe inverse solve agrees with the forward one:")
reach = roll.achievable_angle(113 * u.s)
print(f"  {reach:~.3f} takes {roll.total_time(reach):~.3f}, the target exactly")

in   113 s:  40 deg needs 83.2 s,  +35.8 % -- fits         (can reach 62.0 deg within the allocated duration)
in    90 s:  40 deg needs 83.2 s,   +8.2 % -- fits         (can reach 45.0 deg within the allocated duration)
in    70 s:  40 deg needs 83.2 s,  -15.9 % -- does not fit (can reach 30.3 deg within the allocated duration)

the inverse solve agrees with the forward one:
  61.960 deg takes 113.000 s, the target exactly


### A slew costs swath

Tying the time to the shared mission's ground track speed turns a slew into kilometres of ground the satellite did not image — which is the currency a payload operator actually thinks in, and the reason agility is a budget rather than a specification.

In [11]:
print(f"ground track speed at {mission.altitude:~.0f}:  {mission.ground_track_speed:~.4f}")
for angle in (20 * u.deg, 40 * u.deg, 90 * u.deg):
    duration = roll.total_time(angle)
    print(f"  {angle:~5.0f} costs {duration:~6.1f} and "
          f"{roll.ground_distance(duration):~.0f} of ground track")

ground track speed at 500 km:  7.0592 km / s
     20 deg costs   56.0 s and 396 km of ground track
     40 deg costs   83.2 s and 587 km of ground track
     90 deg costs  151.1 s and 1066 km of ground track


## The document view

`tabulated_agility()` renders the slew table for reading. The requirement check is optional and follows the target duration: give one and the table gains a margin and a verdict, marking in red what does not fit; without one those two columns are not merely hidden but absent, because there is nothing to check against.

That is a deliberate difference from the other reports. `comments` on the delta-V budget hides a column that always exists; here the column cannot be computed at all until you say what you are checking against.

In [12]:
roll.tabulated_agility(target_duration=113 * u.s)

Slew [deg],Limited by,Slew [s],With settling [s],Peak rate [deg/s],Momentum used,Time margin,
5,torque,15.5,35.5,0.6440,87.4%,+218.1%,PASS
10,momentum,22.5,42.5,0.7366,100.0%,+166.2%,PASS
15,momentum,29.2,49.2,0.7366,100.0%,+129.5%,PASS
20,momentum,36.0,56.0,0.7366,100.0%,+101.7%,PASS
40,momentum,63.2,83.2,0.7366,100.0%,+35.8%,PASS
45,momentum,70.0,90.0,0.7366,100.0%,+25.6%,PASS
60,momentum,90.3,110.3,0.7366,100.0%,+2.4%,PASS
90,momentum,131.1,151.1,0.7366,100.0%,-25.2%,FAILS


In [13]:
plain = roll.tabulated_agility()
print("without a target duration, the check is not computed at all:")
print(f"  columns: {list(plain.data.columns)}")

without a target duration, the check is not computed at all:
  columns: ['angle', 'profile', 'slew_time', 'total_time', 'peak_rate', 'momentum_used']


## Limitations

What the agility budget deliberately does not do:

- **No distribution matrix.** The wheel geometry collapses to a projection factor, so nothing here tells you the per-wheel torque or momentum, and nothing checks an individual wheel is inside its own limits. A real ADCS design needs that; a sizing model does not.
- **Rest to rest, one axis at a time.** No coupled manoeuvres, no gyroscopic cross-terms, no products of inertia — the inertia is a single number per axis, not a tensor. A slew about roll and pitch together is not this model.
- **The profile is ideal.** Instantaneous torque reversal at the midpoint, no jerk limit, no actuator lag, and settling is a flat allowance rather than anything derived from the control loop.
- **No disturbance torques.** Gravity gradient, aerodynamic and solar torques are absent. They are what the momentum use factor and torque derating are held back *for*, but the reserve is a policy figure, not a calculation.
- **One failed wheel, and only the halving.** The degraded case assumes the remaining geometry still balances at half capability. A different failure, or a mounting that does not, is not covered.
- **Nothing checks a stated inertia.** An envelope case follows the mass budget; a stated one is a fixed number, and if the spacecraft grows nobody will tell you. That is the trade for being able to state a measured figure at all.